<a href="https://colab.research.google.com/github/routeway-arch/mscijquants-tools/blob/main/msci_candidate_radar_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 株式波動トレード × AI ｜ MSCI 候補レーダー

MSCI 標準指数（ACWI 日本部分）の **採用・除外の候補** を、しきい値までの距離で
ランキング表示するツールです。Google Colab で動き、株価はあなた自身の
**J-Quants API キー** で取得します。

何をするか — 各銘柄の浮動株調整後の時価総額を計算し、MSCI の採用ライン
（GMSR × 1.8）と除外ライン（GMSR × 0.5）と比べて、次の見直しで
「採用されそう／除外されそう」な銘柄を炙り出します。さらに採用・除外時に
発生するパッシブ資金の強制売買を「平均売買代金の何日分」かで見積もります。

## 使い方（4ステップ）

1. J-Quants に登録（API利用には Free を含むいずれかのプラン契約が必要。直近データは有料プラン推奨）。
2. 「設定」セルの GMSR・USDJPY などを、メンバーシップ今月号の数値に合わせる。
3. MSCI ユニバース・フィード（構成・発行株数・浮動株率）を読み込む。
4. 上から順に実行。採用レーダーと除外レーダーが表示されます。

## 今月の確認ポイント（2026年5月見直し）

今回の見直しでは **古河電工(5801)・三井金属(5706)・レゾナック(4004)** が採用、
日本航空ほか14銘柄が除外されました。レーダーがこの3銘柄を「採用ライン超え」で
拾えるかを確認し、ロジックを納得してから次回（8月）の予測に使ってください。

## 注意

- 本ツールは公開された指数ルールを機械的に当てはめて銘柄を並べ替える分析の道具であり、
  特定銘柄の売買を推奨するものではありません。投資判断はご自身の責任で行ってください。
- GMSR・浮動株率・パッシブ資金額は**推定値**です。値によって結果は変わります。
- データはあなたが契約する J-Quants から取得されます。規約の遵守は提供元の規定に従います。
- J-Quants の API キーは第三者に共有・配布しないでください。


## 1. 準備

In [2]:
import os, time, json, getpass
import datetime as dt
import requests
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("ライブラリ読み込み完了")


ライブラリ読み込み完了


## 2. Google ドライブを接続

In [3]:
from google.colab import drive
drive.mount("/content/drive")

WORKDIR = "/content/drive/MyDrive/msci_radar"
os.makedirs(WORKDIR, exist_ok=True)
print("保存先:", WORKDIR)


Mounted at /content/drive
保存先: /content/drive/MyDrive/msci_radar


## 3. 設定

★印はメンバーシップ「今月号」のブリーフィングに記載された数値に差し替えてください。

In [5]:
# ===== MSCI 候補レーダー設定 =====
GMSR_USD        = 2_800_000_000   # ★ GMSR（最小サイズ基準・USD）。今月号の推定値に差し替え
USDJPY          = 155.0           # ★ USD/JPY 為替レート。直近値に更新
ADD_MULTIPLE    = 1.8             # 採用バッファ（SAIR: GMSR × 1.8）
DELETE_MULTIPLE = 0.5             # 除外バッファ（SAIR: GMSR × 0.5）
PASSIVE_AUM_USD = 60_000_000_000  # ★ MSCI日本連動パッシブ資金の推定額（強制フロー試算用）
ADV_DAYS         = 20             # 平均売買代金（ADV）の算出日数
LOOKBACK_DAYS    = 90             # 価格取得の遡及日数（無料プランは 250 以上に）
REQUESTS_PER_MIN = 60             # ★ J-Quantsプランの上限に合わせる（Free:5 / Light:60 / Standard:120 / Premium:500）

# MSCIユニバース・フィードURL（月額メンバー用。空ならCSVアップロード/サンプルで動作）
FEED_URL        = ""

print("設定OK  採用ライン =", f"{GMSR_USD*ADD_MULTIPLE:,.0f} USD",
      " 除外ライン =", f"{GMSR_USD*DELETE_MULTIPLE:,.0f} USD")


設定OK  採用ライン = 5,040,000,000 USD  除外ライン = 1,400,000,000 USD


## 4. J-Quants 認証（V2 / APIキー方式）

J-Quants API V2 は **APIキー方式**です。J-Quants の
[ダッシュボード](https://jpx-jquants.com/ja/dashboard) で発行したAPIキーを入力してください。
APIキーには有効期限がなく、トークンの更新作業は不要です。
キーは保存されず、実行のたびに入力します。

In [6]:
JQ_BASE = "https://api.jquants.com/v2"

# J-Quants ダッシュボードで発行した API キーを入力（保存しません。実行ごとに入力）
API_KEY = getpass.getpass("J-Quants API キーを貼り付けて Enter: ").strip()
HEADERS = {"x-api-key": API_KEY}

# 疎通確認（過去日の1銘柄を取得してキーの有効性を確認）
_t = requests.get(f"{JQ_BASE}/equities/bars/daily",
                  headers=HEADERS, params={"code": "7203", "date": "2024-01-04"}, timeout=30)
assert _t.status_code == 200, f"認証失敗: {_t.status_code} {_t.text}"
print("認証OK（APIキー有効）")


J-Quants API キーを貼り付けて Enter: ··········
認証OK（APIキー有効）


## 5. MSCI ユニバース・フィードの読み込み

必要な列は `code`（4桁）, `name`, `shares`（発行済株式数）, `fif`（浮動株比率の推定 0〜1）,
`msci_member`（現在の構成銘柄なら Y / それ以外 N）。
優先順位は フィードURL → ドライブ内CSV → アップロード → サンプル の順です。

In [7]:
from google.colab import files as colab_files

# ※サンプルは形式確認用のプレースホルダ値です（発行株数・浮動株率は概算）。
#   実運用ではメンバーシップのユニバース・フィードに差し替えてください。
SAMPLE_UNIVERSE = [
    ("5801","古河電気工業",                700_000_000, 0.90, "N"),
    ("5706","三井金属鉱業",                570_000_000, 0.85, "N"),
    ("4004","レゾナック・ホールディングス", 190_000_000, 0.95, "N"),
    ("3626","TIS",                         260_000_000, 0.80, "Y"),
    ("3064","MonotaRO",                    500_000_000, 0.45, "Y"),
    ("6869","シスメックス",                210_000_000, 0.90, "Y"),
    ("9201","日本航空",                    440_000_000, 0.85, "Y"),
    ("7203","トヨタ自動車",             13_000_000_000, 0.60, "Y"),
    ("6758","ソニーグループ",            1_200_000_000, 0.90, "Y"),
    ("8035","東京エレクトロン",            470_000_000, 0.95, "Y"),
]

def normalize(df):
    cols = {c.lower(): c for c in df.columns}
    need = ["code", "shares", "fif", "msci_member"]
    for k in need:
        if k not in cols:
            raise ValueError(f"列 '{k}' が見つかりません。必要な列: code, name, shares, fif, msci_member")
    name_col = cols.get("name")
    out = []
    for _, r in df.iterrows():
        code = str(r[cols["code"]]).strip().split(".")[0][:4]
        if not code or code.lower() == "nan":
            continue
        try:
            shares = float(str(r[cols["shares"]]).replace(",", ""))
            fif    = float(r[cols["fif"]])
        except Exception:
            continue
        member = str(r[cols["msci_member"]]).strip().upper()[:1]
        member = "Y" if member == "Y" else "N"
        name = str(r[name_col]).strip() if name_col else code
        out.append((code, name, shares, fif, member))
    return out

def load_universe():
    local = os.path.join(WORKDIR, "msci_universe.csv")
    if FEED_URL:
        try:
            u = normalize(pd.read_csv(FEED_URL, dtype=str))
            print(f"フィードから {len(u)} 銘柄を取得"); return u, False
        except Exception as e:
            print("フィード取得失敗:", e)
    if os.path.exists(local):
        u = normalize(pd.read_csv(local, dtype=str))
        print(f"ドライブのCSVから {len(u)} 銘柄を読込"); return u, False
    print("MSCIユニバースCSVをアップロードしてください（キャンセルでサンプル動作）")
    up = colab_files.upload()
    if up:
        df = pd.read_csv(list(up.keys())[0], dtype=str)
        df.to_csv(local, index=False)
        u = normalize(df)
        print(f"{len(u)} 銘柄を読込（次回用にドライブへ保存）"); return u, False
    print("※ サンプル10銘柄（プレースホルダ）で動作します")
    return SAMPLE_UNIVERSE, True

universe, is_sample = load_universe()
print("対象:", len(universe), "銘柄  / 現構成:",
      sum(1 for x in universe if x[4] == "Y"), "銘柄")


MSCIユニバースCSVをアップロードしてください（キャンセルでサンプル動作）


※ サンプル10銘柄（プレースホルダ）で動作します
対象: 10 銘柄  / 現構成: 7 銘柄


## 6. 株価・売買代金の取得

J-Quants V2（/equities/bars/daily）から各銘柄の終値と売買代金を取得します。プランのレート上限に合わせて間隔を空けるため、銘柄数によって数分かかります。

In [8]:
REQUEST_INTERVAL = 60.0 / max(REQUESTS_PER_MIN, 1) * 1.1   # 10%の安全マージン

def fetch_bars(code, dfrom, dto):
    rows, pkey = [], None
    while True:
        params = {"code": code, "from": dfrom, "to": dto}
        if pkey:
            params["pagination_key"] = pkey
        resp = None
        for attempt in range(5):
            try:
                resp = requests.get(f"{JQ_BASE}/equities/bars/daily",
                                    headers=HEADERS, params=params, timeout=30)
            except Exception:
                time.sleep(2 ** attempt); continue
            if resp.status_code == 200:
                break
            if resp.status_code == 429:   # レート上限 -> 待機して再試行
                time.sleep(int(resp.headers.get("Retry-After", 5 * (attempt + 1))))
                continue
            return rows
        if resp is None or resp.status_code != 200:
            return rows
        body = resp.json()
        rows.extend(body.get("data", []))
        pkey = body.get("pagination_key")
        if not pkey:
            break
    return rows

today = dt.date.today()
dfrom = (today - dt.timedelta(days=LOOKBACK_DAYS)).isoformat()
dto   = today.isoformat()

est_min = len(universe) * REQUEST_INTERVAL / 60
print(f"{len(universe)} 銘柄を取得します（推定 {est_min:.1f} 分）...")

quotes_by_code, fails = {}, []
for i, (code, name, *_ ) in enumerate(universe, 1):
    q = fetch_bars(code, dfrom, dto)
    if q:
        quotes_by_code[code] = q
    else:
        fails.append(code)
    if i % 20 == 0 or i == len(universe):
        print(f"  取得 {i}/{len(universe)}")
    time.sleep(REQUEST_INTERVAL)
print(f"完了: {len(quotes_by_code)} 銘柄取得 / {len(fails)} 銘柄スキップ")


10 銘柄を取得します（推定 0.2 分）...
  取得 10/10
完了: 10 銘柄取得 / 0 銘柄スキップ


## 7. 候補レーダーの計算

浮動株調整後の時価総額（＝終値 × 発行株数 × 浮動株率 ÷ USDJPY）を求め、
採用ライン・除外ラインとの比率で候補を判定します。

In [10]:
ADD_LINE = GMSR_USD * ADD_MULTIPLE
DEL_LINE = GMSR_USD * DELETE_MULTIPLE

def build_row(code, name, shares, fif, member, quotes):
    df = pd.DataFrame(quotes)
    if df.empty or "Date" not in df.columns:
        return None
    df = df.sort_values("Date")
    close = pd.to_numeric(df.get("C"), errors="coerce").dropna()
    tv    = pd.to_numeric(df.get("Va"), errors="coerce").dropna()
    if close.empty:
        return None
    last_close   = float(close.iloc[-1])
    fif_cap_usd  = last_close * shares * fif / USDJPY
    adv_jpy      = float(tv.tail(ADV_DAYS).mean()) if not tv.empty else float("nan")
    return {
        "コード": code, "銘柄名": name, "現構成": member, "浮動株率": fif,
        "終値": round(last_close, 1),
        "FIF時価(億$)": round(fif_cap_usd / 1e8, 2),
        "_fif_usd": fif_cap_usd, "_adv_jpy": adv_jpy,
        "日付": df["Date"].iloc[-1],
    }

rows = []
for code, name, shares, fif, member in universe:
    if code in quotes_by_code:
        r = build_row(code, name, shares, fif, member, quotes_by_code[code])
        if r:
            rows.append(r)
data = pd.DataFrame(rows)
assert not data.empty, "計算できる銘柄がありません。"

index_total_usd = data.loc[data["現構成"] == "Y", "_fif_usd"].sum()

def passive_flow(fif_usd, adv_jpy):
    if not index_total_usd or pd.isna(adv_jpy) or adv_jpy <= 0:
        return float("nan")
    weight     = fif_usd / index_total_usd
    flow_jpy   = PASSIVE_AUM_USD * weight * USDJPY
    return flow_jpy / adv_jpy   # ADV（売買代金）の何日分か

print(f"基準日: {data['日付'].max()}   採用ライン: {ADD_LINE/1e8:,.1f}億$  除外ライン: {DEL_LINE/1e8:,.1f}億$")
if is_sample:
    print("※ サンプルデータのため数値は参考になりません。実フィードで再実行してください。")


基準日: 2026-05-25   採用ライン: 50.4億$  除外ライン: 14.0億$
※ サンプルデータのため数値は参考になりません。実フィードで再実行してください。


## 8. 採用レーダー（現構成外の銘柄）

In [11]:
add_df = data[data["現構成"] == "N"].copy()
add_df["採用ライン比"] = (add_df["_fif_usd"] / ADD_LINE).round(2)
add_df["強制買い(ADV日数)"] = add_df.apply(
    lambda r: round(passive_flow(r["_fif_usd"], r["_adv_jpy"]), 1), axis=1)

def add_judge(x):
    if x >= 1.0: return "★ 採用ライン超え"
    if x >= 0.8: return "● 接近"
    return "− 圏外"
add_df["判定"] = add_df["採用ライン比"].apply(add_judge)
add_df = add_df.sort_values("採用ライン比", ascending=False).reset_index(drop=True)
add_df.index += 1

cols = ["コード","銘柄名","終値","FIF時価(億$)","採用ライン比","強制買い(ADV日数)","判定"]
display(add_df[cols])


,コード,銘柄名,終値,FIF時価(億$),採用ライン比,強制買い(ADV日数),判定
1,5801,古河電気工業,"58,080.00","2,360.67",46.84,24.20,★ 採用ライン超え
2,5706,三井金属鉱業,"53,330.00","1,666.99",33.08,40.40,★ 採用ライン超え
3,4004,レゾナック・ホールディングス,"19,450.00",226.50,4.49,9.70,★ 採用ライン超え


## 9. 除外レーダー（現構成銘柄）

In [12]:
del_df = data[data["現構成"] == "Y"].copy()
del_df["除外ライン比"] = (del_df["_fif_usd"] / DEL_LINE).round(2)
del_df["強制売り(ADV日数)"] = del_df.apply(
    lambda r: round(passive_flow(r["_fif_usd"], r["_adv_jpy"]), 1), axis=1)

def del_judge(x):
    if x < 1.0: return "★ 除外ライン割れ"
    if x < 1.3: return "● 接近"
    return "− 圏内（安全）"
del_df["判定"] = del_df["除外ライン比"].apply(del_judge)
del_df = del_df.sort_values("除外ライン比", ascending=True).reset_index(drop=True)
del_df.index += 1

cols = ["コード","銘柄名","終値","FIF時価(億$)","除外ライン比","強制売り(ADV日数)","判定"]
display(del_df[cols])

stamp = dt.date.today().isoformat()
add_df[cols[:-3]+["採用ライン比","強制買い(ADV日数)","判定"]].to_csv(
    os.path.join(WORKDIR, f"add_radar_{stamp}.csv"), encoding="utf-8-sig")
del_df[cols[:-3]+["除外ライン比","強制売り(ADV日数)","判定"]].to_csv(
    os.path.join(WORKDIR, f"del_radar_{stamp}.csv"), encoding="utf-8-sig")
print("CSVを保存しました:", WORKDIR)


,コード,銘柄名,終値,FIF時価(億$),除外ライン比,強制売り(ADV日数),判定
1,6869,シスメックス,"1,382.50",16.86,1.20,7.40,● 接近
2,3064,MonotaRO,"1,902.50",27.62,1.97,10.90,− 圏内（安全）
3,3626,TIS,"3,416.00",45.84,3.27,20.50,− 圏内（安全）
4,9201,日本航空,"2,699.50",65.14,4.65,17.50,− 圏内（安全）
5,6758,ソニーグループ,"3,598.00",250.70,17.91,7.20,− 圏内（安全）
6,8035,東京エレクトロン,"52,180.00","1,503.12",107.37,27.30,− 圏内（安全）
7,7203,トヨタ自動車,"3,026.00","1,522.76",108.77,46.60,− 圏内（安全）


CSVを保存しました: /content/drive/MyDrive/msci_radar
